# Qwen2.5-7B 量化全流程

这个 notebook 说明 Qwen2.5-7B 常见量化路线：bitsandbytes 8bit、bitsandbytes 4bit、GGUF、GPTQ、AWQ、EXL2、FP8。

量化不是一个东西。不同量化路线解决的问题不同：有的是训练时低显存加载，有的是本地推理格式，有的是服务端推理压缩。先分清用途，再选择工具。

## 1. 常用程度排名

| 排名 | 方法 | 主要用途 | 先学理由 |
| --- | --- | --- | --- |
| 1 | bitsandbytes 4bit | QLoRA 训练、低显存加载 | 和训练最相关，单卡用户最常用 |
| 2 | GGUF | Ollama、llama.cpp、LM Studio | 本地推理最常见 |
| 3 | bitsandbytes 8bit | 低显存加载、简单推理 | 比 4bit 更接近原精度，使用简单 |
| 4 | AWQ | 服务端 4bit 推理 | vLLM/Transformers 生态常见 |
| 5 | GPTQ | GPU 推理量化 | 模型仓库里常见现成版本 |
| 6 | FP8 | 新 GPU 服务端推理 | H100/H200/B200 等场景重要 |
| 7 | EXL2 | ExLlamaV2 本地 GPU 推理 | 本地速度好，但生态更集中 |

## 2. bitsandbytes 4bit

bitsandbytes 4bit 最常见用途是 QLoRA。它的核心不是生成一个 4bit 发布模型，而是在训练时用 4bit 加载基座模型，再训练 LoRA adapter。

关键参数：

- `load_in_4bit`：开启 4bit 加载。
- `bnb_4bit_quant_type`：常用 `nf4`，适合神经网络权重分布。
- `bnb_4bit_compute_dtype`：计算精度，支持时优先 `bfloat16`。
- `bnb_4bit_use_double_quant`：二次量化，进一步降低显存。

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-7B-Instruct"

bnb_4bit_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# 会下载并加载模型，确认环境后再执行。
# tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
# model_4bit = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config=bnb_4bit_config,
#     device_map="auto",
#     trust_remote_code=True,
# )

## 3. bitsandbytes 8bit

8bit 加载比 4bit 更接近原始精度，但显存占用更高。它适合低显存推理验证，或者想比 fp16/bf16 更省显存但不想压到 4bit 的场景。

常见误区：8bit 加载不是导出 8bit 模型文件，它主要是运行时加载方式。

In [ ]:
bnb_8bit_config = BitsAndBytesConfig(load_in_8bit=True)

# 会下载并加载模型，确认环境后再执行。
# model_8bit = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config=bnb_8bit_config,
#     device_map="auto",
#     trust_remote_code=True,
# )

## 4. GGUF

GGUF 是 llama.cpp、Ollama、LM Studio 常用格式。它适合本地推理，尤其是 Mac、CPU、轻量部署。

常见量化级别：

- `Q4_K_M`：最常用平衡选择，体积小，质量尚可。
- `Q5_K_M`：质量更好，体积更大。
- `Q8_0`：更接近原始质量，但占用高。

如果你训练的是 LoRA adapter，通常要先 merge 到基座模型，再导出 GGUF。

In [ ]:
# llama.cpp 的典型流程如下。这里用字符串保存真实命令，避免 notebook 直接执行大型转换。
merge_model_dir = "../../outputs/qwen2_5_7b/merged"
gguf_fp16_path = "../../outputs/qwen2_5_7b/qwen2_5_7b-f16.gguf"
gguf_q4_path = "../../outputs/qwen2_5_7b/qwen2_5_7b-q4_k_m.gguf"

convert_command = f"python convert_hf_to_gguf.py {merge_model_dir} --outfile {gguf_fp16_path} --outtype f16"
quantize_command = f"./llama-quantize {gguf_fp16_path} {gguf_q4_path} Q4_K_M"

print(convert_command)
print(quantize_command)

## 5. AWQ

AWQ 是激活感知权重量化，偏推理部署。它需要校准数据，校准数据越接近真实请求，量化后的质量越可靠。

关键参数：

- `w_bit`：权重量化位数，常用 4。
- `q_group_size`：分组大小，常用 128。
- `zero_point`：是否使用 zero point。
- `version`：实现版本，例如 GEMM。

In [ ]:
# AutoAWQ 示例，安装 autoawq 后可执行。
# from awq import AutoAWQForCausalLM
# from transformers import AutoTokenizer
#
# quant_config = {"zero_point": True, "q_group_size": 128, "w_bit": 4, "version": "GEMM"}
# model = AutoAWQForCausalLM.from_pretrained(model_name, trust_remote_code=True)
# tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
# model.quantize(tokenizer, quant_config=quant_config)
# model.save_quantized("../../outputs/qwen2_5_7b/awq")
# tokenizer.save_pretrained("../../outputs/qwen2_5_7b/awq")

## 6. GPTQ

GPTQ 是常见离线权重量化格式，主要用于 GPU 推理。它通常不作为继续训练的起点。

关键参数：

- `bits`：量化位数，常见 4。
- `group_size`：分组大小，常见 128。
- `desc_act`：是否使用 activation order，可能影响质量和速度。

In [ ]:
# GPTQ 示例，安装 optimum 和 auto-gptq 后可执行。不同版本 API 可能略有变化。
# from transformers import AutoTokenizer
# from optimum.gptq import GPTQQuantizer
#
# tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
# quantizer = GPTQQuantizer(bits=4, dataset="c4", model_seqlen=2048, group_size=128)
# model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
# quantized_model = quantizer.quantize_model(model, tokenizer)
# quantized_model.save_pretrained("../../outputs/qwen2_5_7b/gptq")

## 7. EXL2

EXL2 是 ExLlamaV2 生态的量化格式，适合单机 GPU 高效推理。它不是 vLLM/Ollama 主线，也不是训练格式。

关键参数是 `bpw`，也就是 bits per weight，表示平均每个权重使用多少 bit。bpw 越高质量越好，占用也越高。

In [ ]:
exl2_command = "python convert.py -i ../../outputs/qwen2_5_7b/merged -o ../../outputs/qwen2_5_7b/exl2 -b 4.0"
print(exl2_command)

## 8. FP8

FP8 常见于服务端推理，尤其是 H100、H200、B200 等新硬件。它通常和 vLLM 等推理框架结合使用。

关键点：

- 老 GPU 不一定支持。
- 要看推理框架版本是否支持目标模型的 FP8 路线。
- `kv_cache_dtype`、`quantization`、`tensor_parallel_size` 会影响显存和吞吐。

In [ ]:
vllm_fp8_command = "vllm serve Qwen/Qwen2.5-7B-Instruct --dtype auto --quantization fp8 --max-model-len 8192"
print(vllm_fp8_command)

## 9. 选择建议

- 要训练：先看 bitsandbytes 4bit，也就是 QLoRA。
- 要本地跑：优先 GGUF，然后用 Ollama 或 llama.cpp。
- 要服务端推理：优先看 vLLM 支持的 AWQ、GPTQ、FP8。
- 要 ExLlamaV2 生态：再考虑 EXL2。
- 不要把训练时量化、推理时量化、文件格式量化混为一谈。